# 04 · CrewAI

**Module 06 · Agent Frameworks**

**CrewAI** thinks in terms of a **team**:

| Idea | What it is |
|---|---|
| **Agent** | A team member with a **role**, a **goal** and a **backstory** |
| **Task** | A job to do, with a description and the expected output |
| **Crew** | The team. It works through the tasks. |
| **Process** | How the work is organised: one task after another (`sequential`), or with a manager (`hierarchical`) |

It reads almost like a job description, which makes it easy to explain to someone who does not write code.

> **A note about notebooks.** Jupyter already runs an event loop, and CrewAI refuses to start a crew with the normal `kickoff()` inside one. So in this notebook we write `await crew.kickoff_async()`. In a normal script, plain `crew.kickoff()` works (the desk file uses it). To call the desk's plain functions from here we use `await asyncio.to_thread(...)`.

In [1]:
import os
import asyncio

# CrewAI sends anonymous usage data by default. We switch that off for this course.
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

from typing import Literal
from pydantic import BaseModel
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

import shop_tools as shop

print("Ready")

Ready


## 1. One agent, one task
An agent gets a role, a goal and a backstory. A task says what to do. A crew runs it.

In [2]:
llm = LLM(model=shop.MODEL, temperature=0)

greeter = Agent(
    role="Friendly support agent",
    goal="Make customers feel looked after",
    backstory="You work at an online shop and write short, warm replies.",
    llm=llm,
)

welcome_task = Task(
    description="Write a welcome message for a customer who has just placed their first order.",
    expected_output="One short sentence",
    agent=greeter,
)

crew = Crew(agents=[greeter], tasks=[welcome_task], process=Process.sequential)
result = await crew.kickoff_async()

print(result.raw)
print()
print("Tokens used:", result.token_usage.total_tokens)

Thank you for your order! We're thrilled to have you with us and can't wait for you to enjoy your purchase!

Tokens used: 117


## 2. Tools, and a team of two
Tools work as before: our plain functions. CrewAI wraps them with `tool("name")(function)`, and the docstring is the description.

Now a **team of two**. The Orders specialist finds the facts. A second agent turns the facts into a friendly reply. The second task receives the first task's result through `context`.

In [3]:
# Wrap our plain functions as CrewAI tools. The docstring tells the agent what each tool does.
lookup_order = tool("lookup_order")(shop.lookup_order)
search_policies = tool("search_policies")(shop.search_policies)
check_refund_eligibility = tool("check_refund_eligibility")(shop.check_refund_eligibility)
issue_refund = tool("issue_refund")(shop.issue_refund)


def make_specialists():
    """Build the team. Each agent has a role, a goal, and a backstory (our instructions)."""
    llm = LLM(model=shop.MODEL, temperature=0)
    return {
        "orders": Agent(
            role="Orders specialist",
            goal="Answer questions about where an order is and what is in it",
            backstory=shop.ORDERS_INSTRUCTIONS, tools=[lookup_order], llm=llm),
        "policy": Agent(
            role="Policy specialist",
            goal="Answer questions about shop rules: shipping costs, returns, warranty and payments",
            backstory=shop.POLICY_INSTRUCTIONS, tools=[search_policies], llm=llm),
        "refunds": Agent(
            role="Refunds specialist",
            goal="Handle requests from customers who want their money back",
            backstory=shop.REFUNDS_INSTRUCTIONS, tools=[check_refund_eligibility, issue_refund], llm=llm),
    }

In [4]:
orders_specialist = make_specialists()["orders"]

writer = Agent(
    role="Customer reply writer",
    goal="Turn facts into a short, friendly reply",
    backstory="You write 2 sentence replies to customers. You never invent facts.",
    llm=LLM(model=shop.MODEL, temperature=0),
)

facts_task = Task(
    description="Look up order A1002 and list the facts: the item, the status and the expected delivery date.",
    expected_output="A short list of facts",
    agent=orders_specialist,
)

reply_task = Task(
    description="Write the reply to a customer who asked where order A1002 is.",
    expected_output="2 short sentences, no subject line, no signature",
    agent=writer,
    context=[facts_task],  # this task can see the result of the first one
)

shop.reset_shop()
crew = Crew(agents=[orders_specialist, writer], tasks=[facts_task, reply_task], process=Process.sequential)
result = await crew.kickoff_async()

print("Tools that ran:", shop.TOOL_CALLS)
print("Reply:", result.raw)
print("Tokens used:", result.token_usage.total_tokens)

Tools that ran: ['lookup_order']
Reply: Your order A1002 for the running shoes is currently in transit. It is expected to be delivered by November 25, 2026.
Tokens used: 605


That is the CrewAI way: tasks run in order, and each can use the previous result. It is natural for **pipelines** (research, then write, then review).

## 3. The support desk as a crew
For the desk we need to *choose* a specialist first. We use a one-agent **triage crew** whose answer is turned into a typed `Route` (`output_pydantic`). Then the chosen specialist's crew writes the reply. This is the code from `crewai_desk.py`.

In [5]:
class Route(BaseModel):
    specialist: Literal["orders", "policy", "refunds"]


def choose_specialist(email):
    """A one-agent crew whose only job is to pick the specialist. Returns the name and the tokens used."""
    triage_agent = Agent(
        role="Triage agent",
        goal="Decide which specialist should handle a customer email",
        backstory="orders = where an order is or what is in it. "
                  "policy = shop rules such as shipping, warranty or payments. "
                  "refunds = the customer wants their money back.",
        llm=LLM(model=shop.MODEL, temperature=0))
    triage_task = Task(
        description="Pick the specialist for this customer email:\n" + email,
        expected_output="The name of one specialist",
        agent=triage_agent,
        output_pydantic=Route)  # CrewAI turns the answer into a Route for us

    result = Crew(agents=[triage_agent], tasks=[triage_task], process=Process.sequential).kickoff()
    return result.pydantic.specialist, result.token_usage.total_tokens


def answer_email(specialist_name, email):
    """A one-agent crew: the chosen specialist writes the reply. Returns the reply and the tokens used."""
    specialist = make_specialists()[specialist_name]
    reply_task = Task(
        description="Reply to this customer email:\n" + email,
        expected_output="Only the reply text for the customer: 2 short sentences, no subject line, no greeting, no signature.",
        agent=specialist)

    result = Crew(agents=[specialist], tasks=[reply_task], process=Process.sequential).kickoff()
    return result.raw, result.token_usage.total_tokens

In [6]:
emails = [
    "Hi, can you tell me where my order A1002 is?",
    "How much does express shipping cost?",
    "The left earbud on my headphones (order A1001) stopped working. I would like my money back.",
]

for email in emails:
    shop.reset_shop()
    specialist, router_tokens = await asyncio.to_thread(choose_specialist, email)
    reply, agent_tokens = await asyncio.to_thread(answer_email, specialist, email)

    print("Email:", email)
    print("  Handled by:", specialist)
    print("  Tools that ran:", shop.TOOL_CALLS)
    print("  Refunds issued:", shop.REFUNDS)
    print("  Reply:", reply)
    print("  Tokens:", router_tokens + agent_tokens)
    print()

Email: Hi, can you tell me where my order A1002 is?
  Handled by: orders
  Tools that ran: ['lookup_order']
  Refunds issued: []
  Reply: Your order A1002 for running shoes is currently in transit. It is expected to be delivered by November 25, 2026.
  Tokens: 806



Email: How much does express shipping cost?
  Handled by: policy
  Tools that ran: ['search_policies']
  Refunds issued: []
  Reply: Express shipping always costs $12. Standard shipping is free on orders over $50 and costs $4.99 below that.
  Tokens: 948



Email: The left earbud on my headphones (order A1001) stopped working. I would like my money back.
  Handled by: refunds
  Tools that ran: ['check_refund_eligibility', 'issue_refund']
  Refunds issued: [{'order_id': 'A1001', 'amount': 79.0}]
  Reply: A refund of $79.00 has been issued for your order A1001. You should see the amount reflected in your account shortly.
  Tokens: 1296



## 4. The "manager" way: `Process.hierarchical`
CrewAI's most talked-about feature is a **manager**: you give it a whole team and one task, and a manager agent decides who does what. No router to write. Let us try it on the same three emails.

In [7]:
for email in emails:
    shop.reset_shop()
    team = make_specialists()  # a fresh team and manager for every email, so the token count is per email
    manager_llm = LLM(model=shop.MODEL, temperature=0)

    task = Task(
        description="Reply to this customer email:\n" + email,
        expected_output="Only the reply text for the customer: 2 short sentences, no subject line, no greeting, no signature.",
    )
    crew = Crew(
        agents=[team["orders"], team["policy"], team["refunds"]],
        tasks=[task],
        process=Process.hierarchical,
        manager_llm=manager_llm,
    )
    result = await crew.kickoff_async()

    print("Email:", email)
    print("  Tools that ran:", shop.TOOL_CALLS)
    print("  Refunds issued:", shop.REFUNDS)
    print("  Reply:", result.raw)
    print("  Tokens:", result.token_usage.total_tokens)
    print()

Email: Hi, can you tell me where my order A1002 is?
  Tools that ran: ['lookup_order']
  Refunds issued: []
  Reply: Your order A1002 for running shoes is currently in transit. The expected delivery date is November 25, 2026.
  Tokens: 2692



Email: How much does express shipping cost?
  Tools that ran: []
  Refunds issued: []
  Reply: I'm unable to provide the express shipping cost at this moment. Please check our website or contact the shipping department for accurate information.
  Tokens: 4752



Email: The left earbud on my headphones (order A1001) stopped working. I would like my money back.
  Tools that ran: ['check_refund_eligibility', 'issue_refund']
  Refunds issued: [{'order_id': 'A1001', 'amount': 79.0}]
  Reply: A refund of $79.00 has been issued for your order A1001 due to the malfunctioning left earbud. You should see the amount reflected in your account shortly.
  Tokens: 4498



Compare this with the previous run. Look at **three things**:
1. **Tokens.** The manager talks to the team in extra steps, so the cost is usually several times higher.
2. **Tools that ran.** Did the shipping answer come from the policy tool? If no tool ran, the reply was not grounded in the shop's policy, however confident it sounds.
3. **The reply.** Is it still short and free of placeholders?

This is the trade-off with "autonomous" teams: less code, **less control, and a bigger bill**. For our desk, a small explicit router (section 3) is cheaper and easier to test.

## 5. What it costs
- **Extra prompts and tokens.** The role, goal and backstory become long hidden prompts, so each call is bigger.
- **Manager mode is hard to predict.** The routing happens inside the model.
- **Heavy install.** CrewAI pulls in many packages, which is why this module has its own environment.
- **Anonymous telemetry is on by default.** We switched it off in the setup cell.

**Pick it when** the work looks like a team doing a job in steps (research, write, review) and you want the code to read like a job description.

Next: all three, the same desk, side by side.